# ShiftGuard10 — Project Walkthrough

**EE708 Robust Image Classification Challenge | IIT Kanpur**

**Result:** Rank 1 / 30 teams | Macro F1 = 0.9488

This notebook walks through every design decision in the winning pipeline:
1. Problem setup and class imbalance analysis
2. WideResNet-28-10 architecture
3. Balanced Softmax Loss
4. AutoAugment + MixUp/CutMix
5. Cosine LR Warmup + Nesterov SGD
6. Stochastic Weight Averaging (SWA)
7. 3-Seed Ensemble + 31-View TTA

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import torch
import torch.nn.functional as F

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded.')

---
## 1. Problem Setup — Distribution Shift

The competition presents a **covariate shift**: `P_train(x)` ≠ `P_test(x)` while labels `y` follow the same 10 CIFAR classes. The test images come from a different capture/processing pipeline.

This rules out:
- Simple data augmentation (memorises train distribution)
- Pretrained ImageNet features (different domain, and competition rules ban external data)
- Pure accuracy optimisation (accuracy = macro F1 only when classes are balanced, which they aren't)

In [ ]:
# Simulated class distribution from the competition dataset
CLASS_NAMES = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

# Competition had imbalanced classes (representative counts)
class_counts = np.array([3100, 2700, 2650, 2400, 3050, 2350, 3000, 2900, 3200, 3050])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(CLASS_NAMES)))

bars = axes[0].bar(CLASS_NAMES, class_counts, color=colors, edgecolor='white')
axes[0].bar_label(bars, fmt='%d', padding=3, fontsize=8)
axes[0].set_title('Training Class Distribution', fontweight='bold')
axes[0].tick_params(axis='x', rotation=35)
axes[0].axhline(class_counts.mean(), color='navy', linestyle='--', linewidth=1.2, label=f'Mean = {class_counts.mean():.0f}')
axes[0].legend()

# Macro F1 vs accuracy gap illustration
imbalance_ratio = np.array([1.0, 1.1, 1.3, 1.6, 2.0, 2.5, 3.0, 4.0, 5.0])
f1_gap = 0.025 * np.log(imbalance_ratio)  # illustrative
axes[1].plot(imbalance_ratio, f1_gap * 100, 'o-', color='crimson', linewidth=2)
axes[1].fill_between(imbalance_ratio, f1_gap * 100, alpha=0.15, color='crimson')
axes[1].set_xlabel('Max/Min Class Ratio (imbalance severity)')
axes[1].set_ylabel('Accuracy − Macro F1 gap (%)')
axes[1].set_title('Why Macro F1 Matters', fontweight='bold')

plt.tight_layout()
plt.show()
print('Our dataset: max/min ratio ≈', round(class_counts.max()/class_counts.min(), 2),
      '— significant enough to hurt minority-class recall if not addressed.')

---
## 2. WideResNet-28-10 Architecture

**Why not ResNet-50?** Its 7×7 stem + MaxPool reduces 32×32 → 8×8 in the first two layers — catastrophic for small images.

**WRN-28-10:** 3×3 stem, 3 residual groups at strides [1, 2, 2], global avg pool. No initial downsampling.

```
Input  (B, 3, 32, 32)
Conv1  (B, 16, 32, 32)  — 3×3, stride 1
Group1 (B, 160, 32, 32) — 4× BasicBlock, stride 1
Group2 (B, 320, 16, 16) — 4× BasicBlock, stride 2
Group3 (B, 640,  8,  8) — 4× BasicBlock, stride 2
BN+ReLU → GAP → Linear(640, 10)
```

In [ ]:
from src.models.wideresnet import wrn_28_10

model = wrn_28_10(num_classes=10)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'WideResNet-28-10 parameters: {n_params:,} ({n_params/1e6:.1f}M)')

# Layer-by-layer parameter breakdown
layer_params = {
    'Conv1 (stem)':  sum(p.numel() for p in model.conv1.parameters()),
    'Group 1':       sum(p.numel() for p in model.group1.parameters()),
    'Group 2':       sum(p.numel() for p in model.group2.parameters()),
    'Group 3':       sum(p.numel() for p in model.group3.parameters()),
    'BN + Linear':   sum(p.numel() for p in model.bn.parameters()) +
                     sum(p.numel() for p in model.fc.parameters()),
}

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(list(layer_params.keys()), [v/1e6 for v in layer_params.values()],
               color=['#2c7bb6','#abd9e9','#ffffbf','#fdae61','#d7191c'])
ax.bar_label(bars, fmt='%.2fM', padding=4, fontsize=9)
ax.set_xlabel('Parameters (millions)')
ax.set_title('WideResNet-28-10 Parameter Distribution', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Balanced Softmax Loss

Standard cross-entropy treats all classes equally in the denominator:
```
p(c|x) = exp(f_c) / Σ_j exp(f_j)
```
Balanced Softmax adjusts by class frequency `n_c`:
```
p(c|x) = n_c · exp(f_c) / Σ_j n_j · exp(f_j)
```
Equivalent: add `log(n_c)` to each class logit before standard cross-entropy.

In [ ]:
from src.loss import BalancedSoftmaxLoss

# Demonstrate the logit-shift effect
cls_counts = np.array([3100, 2700, 2650, 2400, 3050, 2350, 3000, 2900, 3200, 3050])
log_freq_bias = np.log(cls_counts / cls_counts.sum())

# Suppose a model produces equal logits (0.0) for all classes
equal_logits   = np.zeros(10)
adjusted_logits = equal_logits + log_freq_bias
std_probs = np.exp(equal_logits) / np.exp(equal_logits).sum()
bal_probs = np.exp(adjusted_logits) / np.exp(adjusted_logits).sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

x = np.arange(len(CLASS_NAMES))
w = 0.35
axes[0].bar(x - w/2, std_probs * 100, w, label='Standard Softmax', color='#4e79a7')
axes[0].bar(x + w/2, bal_probs * 100, w, label='Balanced Softmax', color='#e15759')
axes[0].set_xticks(x)
axes[0].set_xticklabels(CLASS_NAMES, rotation=35, ha='right', fontsize=8)
axes[0].set_ylabel('Predicted probability (%) with equal logits')
axes[0].set_title('Balanced Softmax shifts p toward minority classes', fontweight='bold')
axes[0].legend()

axes[1].bar(CLASS_NAMES, log_freq_bias, color='#f28e2b', edgecolor='white')
axes[1].set_title('log(n_c) bias term added to each class logit', fontweight='bold')
axes[1].set_ylabel('Logit adjustment (log n_c / Σ n_j)')
axes[1].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.show()
print('Dog (lowest count) gets the largest positive bias:', round(log_freq_bias[5], 4))
print('Ship (highest count) gets the smallest bias:', round(log_freq_bias[8], 4))

---
## 4. AutoAugment Visualisation

The CIFAR10 AutoAugment policy applies two operations per image from a learned set of 25 sub-policies.
Operations include: Rotate, ShearX/Y, TranslateX/Y, Equalize, AutoContrast, Posterize, Solarize,
Color, Contrast, Brightness, Sharpness.

In [ ]:
from torchvision.transforms import AutoAugment, AutoAugmentPolicy, ToTensor, Normalize, Compose
from PIL import Image
import numpy as np

# Create a synthetic test image (solid blocks) to show augmentation effects
synthetic = np.zeros((32, 32, 3), dtype=np.uint8)
synthetic[:16, :16] = [255,   0,   0]   # top-left: red
synthetic[:16, 16:] = [  0, 200,   0]   # top-right: green
synthetic[16:, :16] = [  0,   0, 255]   # bot-left: blue
synthetic[16:, 16:] = [255, 255,   0]   # bot-right: yellow
pil_img = Image.fromarray(synthetic)

aa = AutoAugment(AutoAugmentPolicy.CIFAR10)

fig, axes = plt.subplots(2, 6, figsize=(13, 5))
axes[0, 0].imshow(pil_img)
axes[0, 0].set_title('Original', fontsize=8)
axes[0, 0].axis('off')

for i in range(1, 12):
    r, c = i // 6, i % 6
    aug_img = aa(pil_img)
    axes[r][c].imshow(aug_img)
    axes[r][c].set_title(f'View {i}', fontsize=8)
    axes[r][c].axis('off')

fig.suptitle('AutoAugment CIFAR10 Policy — 11 sample views of the same image',
             fontweight='bold', fontsize=11)
plt.tight_layout()
plt.show()

---
## 5. MixUp and CutMix Visualisation

In [ ]:
from src.utils import mixup_data, cutmix_data

# Create two synthetic 32×32 images as tensors
img_a = torch.zeros(1, 3, 32, 32)
img_a[0, 0, :, :] = 1.0   # red channel

img_b = torch.zeros(1, 3, 32, 32)
img_b[0, 2, :, :] = 1.0   # blue channel

label_a = torch.tensor([0])
label_b = torch.tensor([1])

batch  = torch.cat([img_a, img_b])
labels = torch.cat([label_a, label_b])

# MixUp with λ=0.7
torch.manual_seed(42)
lam = 0.7
mixed_x  = lam * batch[0:1] + (1 - lam) * batch[1:2]

# CutMix manually
cutmix_x = batch[0:1].clone()
cutmix_x[0, :, 16:, 16:] = batch[1:2, :, 16:, 16:]

def t2img(t):
    return t.squeeze().permute(1, 2, 0).clamp(0, 1).numpy()

fig, axes = plt.subplots(1, 5, figsize=(13, 3))
for ax, (img, title) in zip(axes, [
    (img_a, 'Image A\n(Red)'),
    (img_b, 'Image B\n(Blue)'),
    (None,  'λ = 0.7'),
    (mixed_x, f'MixUp\n0.7A + 0.3B'),
    (cutmix_x, 'CutMix\nA with B patch'),
]):
    if img is None:
        ax.text(0.5, 0.5, title, ha='center', va='center', fontsize=12,
                transform=ax.transAxes)
        ax.axis('off')
    else:
        ax.imshow(t2img(img))
        ax.set_title(title, fontsize=9)
        ax.axis('off')

plt.suptitle('MixUp vs CutMix — Input Space Interpolation', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Learning Rate Schedule — Cosine Warmup

In [ ]:
total_epochs  = 300
warmup_epochs = 5
swa_start     = 225
base_lr       = 0.1
swa_lr        = 0.005

lrs = []
for ep in range(total_epochs):
    if ep < warmup_epochs:
        lr = base_lr * (ep + 1) / warmup_epochs
    elif ep < swa_start:
        prog = (ep - warmup_epochs) / (swa_start - warmup_epochs)
        lr = base_lr * 0.5 * (1 + np.cos(np.pi * prog))
    else:
        lr = swa_lr
    lrs.append(lr)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(range(total_epochs), lrs, color='#2c7bb6', linewidth=2)
ax.axvspan(0,          warmup_epochs, alpha=0.12, color='green',  label='Warmup (5 ep)')
ax.axvspan(warmup_epochs, swa_start,  alpha=0.08, color='blue',   label='Cosine decay')
ax.axvspan(swa_start,  total_epochs,  alpha=0.12, color='orange', label='SWA (epoch 225→300)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Schedule: Cosine Warmup + SWA', fontweight='bold')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

---
## 7. 3-Seed Ensemble Variance Reduction

Why 3 seeds? Each seed leads to a different local minimum. Averaging their probability
estimates reduces prediction variance. The gain is largest for the first few models.

In [ ]:
# Theoretical variance reduction from averaging K models
# Var(ensemble) = (sigma^2 + (K-1)*rho*sigma^2) / K
# where sigma^2 = individual model variance, rho = correlation between models

K_range = np.arange(1, 11)
sigma2 = 1.0
rho    = 0.6  # typical inter-model correlation in deep ensembles

var_ensemble = (sigma2 + (K_range - 1) * rho * sigma2) / K_range
var_relative = var_ensemble / sigma2 * 100  # as % of single-model variance

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(K_range, var_relative, 'o-', color='#d62728', linewidth=2, markersize=7)
ax.axvline(3, color='#2ca02c', linestyle='--', linewidth=1.5, label='Our ensemble (K=3)')
ax.fill_between(K_range, var_relative, 100, alpha=0.12, color='#d62728')
ax.set_xlabel('Number of ensemble members (K)')
ax.set_ylabel('Variance (% of single-model)')
ax.set_title('Ensemble Variance Reduction (ρ=0.6 between models)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()
print(f'K=1: 100% variance | K=3: {var_relative[2]:.1f}% | K=5: {var_relative[4]:.1f}% | K=10: {var_relative[9]:.1f}%')
print('Diminishing returns beyond K=3 justify the 3-seed choice.')

---
## 8. 31-View TTA Effect

In [ ]:
# Simulate variance reduction from TTA averaging
np.random.seed(42)
n_views_range = np.arange(1, 35)

# Simulate per-view noise: Gaussian around true probability
true_prob = 0.94
noise_std  = 0.04

std_of_mean = noise_std / np.sqrt(n_views_range)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(n_views_range, std_of_mean * 100, 'o-', color='#1f77b4', linewidth=2, markersize=5)
axes[0].axvline(31, color='#d62728', linestyle='--', linewidth=1.5, label='Our TTA (31 views)')
axes[0].set_xlabel('TTA views')
axes[0].set_ylabel('Std of probability estimate (%)')
axes[0].set_title('TTA Variance vs. Number of Views', fontweight='bold')
axes[0].legend()

# Show that 31 views is near the diminishing-returns knee
gain = std_of_mean[0] - std_of_mean  # reduction from 1 view
marginal_gain = np.diff(gain)
axes[1].bar(n_views_range[1:], np.abs(marginal_gain) * 100, color='#ff7f0e', edgecolor='white')
axes[1].axvline(31, color='#d62728', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('TTA views')
axes[1].set_ylabel('Marginal variance reduction (%)')
axes[1].set_title('Marginal Gain from Each Additional View', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 9. Summary & Ablation Table

| Technique | Val Macro F1 | Δ |
|---|---|---|
| Baseline WRN (cross-entropy) | 0.912 | — |
| + Balanced Softmax | 0.924 | +1.2% |
| + AutoAugment | 0.934 | +1.0% |
| + MixUp/CutMix | 0.941 | +0.7% |
| + SWA | 0.947 | +0.6% |
| + 3-seed ensemble | 0.950 | +0.3% |
| + 31-view TTA | **0.950** (val) | +TTA on test |
| **Final (test leaderboard)** | **0.9488** | **Rank 1 / 30** |

In [ ]:
techniques = [
    'Baseline WRN\n(CE loss)', '+ Balanced\nSoftmax',
    '+ AutoAugment', '+ MixUp/\nCutMix', '+ SWA',
    '+ 3-seed\nEnsemble', '+ 31-view\nTTA (test)'
]
f1_scores = [0.912, 0.924, 0.934, 0.941, 0.947, 0.950, 0.9488]
colors = ['#aec7e8'] * 6 + ['#2ca02c']

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(techniques, f1_scores, color=colors, edgecolor='white', width=0.65)
ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=9, fontweight='bold')
ax.set_ylim(0.88, 0.965)
ax.set_ylabel('Macro F1 Score')
ax.set_title('Ablation Study — Cumulative Technique Stack', fontsize=13, fontweight='bold')
ax.axhline(0.9488, color='#d62728', linestyle='--', linewidth=1.2, label='Final test score: 0.9488')
ax.legend()
plt.tight_layout()
plt.show()